In [3]:
import pandas as pd

# Load data
df = pd.read_csv("train.csv")

# Check the data
print(df.head())
# print(df.columns)
# print(df.isnull().sum())

   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2017-152156  08/11/2017  11/11/2017    Second Class    CG-12520   
1       2  CA-2017-152156  08/11/2017  11/11/2017    Second Class    CG-12520   
2       3  CA-2017-138688  12/06/2017  16/06/2017    Second Class    DV-13045   
3       4  US-2016-108966  11/10/2016  18/10/2016  Standard Class    SO-20335   
4       5  US-2016-108966  11/10/2016  18/10/2016  Standard Class    SO-20335   

     Customer Name    Segment        Country             City       State  \
0      Claire Gute   Consumer  United States        Henderson    Kentucky   
1      Claire Gute   Consumer  United States        Henderson    Kentucky   
2  Darrin Van Huff  Corporate  United States      Los Angeles  California   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale     Florida   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale     Florida   

   Postal Code Region       Product ID         Cat

In [5]:
print("Shape:", df.shape)
print("\nMissing Values:")
print(df.isnull().sum())

Shape: (9800, 18)

Missing Values:
Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country           0
City              0
State             0
Postal Code      11
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
dtype: int64


In [7]:
# Fix date column
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True)

# Add Month and Year columns
df["Month"] = df["Order Date"].dt.to_period("M")
df["Year"] = df["Order Date"].dt.year

print("✅ Dates fixed!")
print(df[["Order Date", "Month", "Year"]].head())

✅ Dates fixed!
  Order Date    Month  Year
0 2017-11-08  2017-11  2017
1 2017-11-08  2017-11  2017
2 2017-06-12  2017-06  2017
3 2016-10-11  2016-10  2016
4 2016-10-11  2016-10  2016


In [9]:
# KPI 1 - Total Revenue
total_revenue = df["Sales"].sum().round(2)
print("💰 Total Revenue: $", total_revenue)

# KPI 2 - Revenue by Region
region_sales = df.groupby("Region")["Sales"].sum().round(2).reset_index()
region_sales.columns = ["Region", "Total Sales"]
print("\n📍 Revenue by Region:")
print(region_sales)

# KPI 3 - Top 5 Products
top_products = df.groupby("Product Name")["Sales"].sum().round(2)\
                 .sort_values(ascending=False).head(5).reset_index()
top_products.columns = ["Product Name", "Total Sales"]
print("\n🏆 Top 5 Products:")
print(top_products)

# KPI 4 - Monthly Revenue
monthly_sales = df.groupby("Month")["Sales"].sum().round(2).reset_index()
monthly_sales.columns = ["Month", "Total Sales"]
monthly_sales["Month"] = monthly_sales["Month"].astype(str)
print("\n📅 Monthly Revenue:")
print(monthly_sales.head())

# KPI 5 - Revenue by Category
category_sales = df.groupby("Category")["Sales"].sum().round(2).reset_index()
category_sales.columns = ["Category", "Total Sales"]
print("\n📦 Revenue by Category:")
print(category_sales)

💰 Total Revenue: $ 2261536.78

📍 Revenue by Region:
    Region  Total Sales
0  Central    492646.91
1     East    669518.73
2    South    389151.46
3     West    710219.68

🏆 Top 5 Products:
                                        Product Name  Total Sales
0              Canon imageCLASS 2200 Advanced Copier     61599.82
1  Fellowes PB500 Electric Punch Plastic Comb Bin...     27453.38
2  Cisco TelePresence System EX90 Videoconferenci...     22638.48
3       HON 5400 Series Task Chairs for Big and Tall     21870.58
4         GBC DocuBind TL300 Electric Binding System     19823.48

📅 Monthly Revenue:
     Month  Total Sales
0  2015-01     14205.71
1  2015-02      4519.89
2  2015-03     55205.80
3  2015-04     27906.86
4  2015-05     23644.30

📦 Revenue by Category:
          Category  Total Sales
0        Furniture    728658.58
1  Office Supplies    705422.33
2       Technology    827455.87


In [11]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows

wb = Workbook()

# Helper function to write a dataframe with formatted headers
def write_df(ws, df, start_row, title):
    ws.cell(row=start_row, column=1, value=title).font = Font(bold=True, size=12)
    for col_num, col_name in enumerate(df.columns, 1):
        cell = ws.cell(row=start_row+1, column=col_num, value=col_name)
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="2E75B6")
        cell.alignment = Alignment(horizontal="center")
    for row in dataframe_to_rows(df, index=False, header=False):
        ws.append(row)

# Sheet 1 - Summary
ws1 = wb.active
ws1.title = "Summary"
ws1["A1"] = "Sales Report - Summary"
ws1["A1"].font = Font(bold=True, size=14)
ws1["A3"] = "Total Revenue:"
ws1["B3"] = total_revenue
ws1["A3"].font = Font(bold=True)

# Sheet 2 - Region Sales
ws2 = wb.create_sheet("Region Sales")
write_df(ws2, region_sales, 1, "Revenue by Region")

# Sheet 3 - Top Products
ws3 = wb.create_sheet("Top Products")
write_df(ws3, top_products, 1, "Top 5 Products")

# Sheet 4 - Monthly Sales
ws4 = wb.create_sheet("Monthly Sales")
write_df(ws4, monthly_sales, 1, "Monthly Revenue")

# Sheet 5 - Category Sales
ws5 = wb.create_sheet("Category Sales")
write_df(ws5, category_sales, 1, "Revenue by Category")

# Save
wb.save("Sales_Report.xlsx")
print("✅ Sales_Report.xlsx has been saved!")

✅ Sales_Report.xlsx has been saved!
